# 01 — Synthetic Data Generation

Generates synthetic scam messages across 5 languages (EN, Singlish, MS, TA, ZH)
using slot-filled templates and NLLB-200 machine translation.

**Output:** `/kaggle/working/synthetic_dataset.csv`

| Step | Description |
|---|---|
| 1 | Setup & paths |
| 2 | Install dependencies |
| 3 | Imports |
| 4 | Slot values & templates |
| 5 | Generate English & Singlish |
| 6 | Load NLLB-200 & translate |
| 7 | Merge, dedup, save |


## 1. Setup

In [1]:
import os

BASE_DIR  = "/kaggle/working"
DATA_DIR  = os.path.join(BASE_DIR, "data")
MODEL_DIR = os.path.join(BASE_DIR, "models")
REPORT_DIR = os.path.join(BASE_DIR, "reports")

for d in [DATA_DIR, MODEL_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Dirs ready")
print("BASE_DIR:", BASE_DIR)


Dirs ready
BASE_DIR: /kaggle/working


## 2. Install dependencies

In [2]:
# install translation dependencies
!pip install -q transformers sentencepiece torch tqdm


## 3. Imports

In [3]:
import random
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

random.seed(42)
print("imports done")


imports done


## 4. Slot values, templates, Singlish templates

In [4]:
# slot values — sampled randomly per message
SLOTS = {
    "bank":      ["DBS", "OCBC", "UOB", "PayNow", "POSB", "Singpass", "CPF",
                  "Standard Chartered", "Citibank", "HSBC", "Maybank", "Trust Bank"],
    "amount":    ["50", "100", "150", "200", "250", "300", "500",
                  "750", "1000", "1500", "2000", "3000", "5000"],
    "frequency": ["daily", "weekly", "per session", "per hour",
                  "every day", "per task", "per shift"],
    "action":    ["suspended", "locked", "frozen", "flagged", "deactivated",
                  "restricted", "temporarily disabled", "placed on hold", "blocked"],
    "activity":  ["login", "transaction", "transfer", "withdrawal",
                  "access attempt", "payment", "fund movement", "sign-in"],
    "service":   ["PayNow access", "online banking", "debit card", "e-wallet",
                  "account access", "internet banking", "mobile app", "card services"],
    "urgency":   [
        "Act now",
        "Verify immediately",
        "Do not ignore this message",
        "Respond within 24 hours",
        "Failure to act will result in permanent closure",
        "Click the link to resolve",
        "Contact support immediately",
        "Immediate action required",
        "Your account will be terminated",
        "Reply now to avoid penalty",
        "Respond before midnight",
        "This is your final notice",
        "Failure to respond will result in legal action",
        "Your access will be revoked",
        "Confirm your details to avoid disruption",
    ],
    "role":      ["data entry clerk", "social media assistant", "survey taker",
                  "online reviewer", "product tester", "content moderator",
                  "listing assistant", "order processor"],
    "platform":  ["Telegram", "WhatsApp", "Signal", "WeChat", "Line", "Instagram", "Facebook"],
    "penalty":   ["SGD {fine} fine", "legal action", "account termination",
                  "credit score impact", "court summons", "blacklisting"],
    "area":      ["Jurong", "Tampines", "Woodlands", "Punggol", "Clementi",
                  "Bishan", "Bedok", "Toa Payoh", "Ang Mo Kio", "Sengkang"],
    "fine":      ["200", "500", "1000", "2000", "5000"],
}

# 12 scam types x 5 templates = 60 base English strings
# taxonomy from SPF Annual Scams and Cybercrime Brief 2025
TEMPLATES = {
    "phishing": [
        "Your {bank} account has been {action}. {urgency}.",
        "Suspicious {activity} detected on your {bank} account. {urgency}.",
        "Your {bank} {service} is blocked. Verify identity to restore. {urgency}.",
        "Urgent security alert from {bank}. Confirm your details now. {urgency}.",
        "Unauthorised {activity} on your {bank} account. {urgency}.",
    ],
    "job_scam": [
        "Earn SGD {amount} {frequency} working from home. {urgency}.",
        "Online job available. SGD {amount} {frequency} payout guaranteed. {urgency}.",
        "{platform} task job. Commission SGD {amount} {frequency}. {urgency}.",
        "Part-time {role} needed urgently. Earn SGD {amount} {frequency}. {urgency}.",
        "No experience needed. {role} earns SGD {amount} daily. {urgency}.",
    ],
    "investment": [
        "Invest SGD {amount} and double your money in 30 days. {urgency}.",
        "Crypto scheme guarantees SGD {amount} returns weekly. {urgency}.",
        "Start with SGD {amount} and earn passive income {frequency}. {urgency}.",
        "Exclusive investment group. SGD {amount} entry. Guaranteed returns. {urgency}.",
        "Limited slots for high-yield fund. Minimum SGD {amount}. {urgency}.",
    ],
    "ecommerce": [
        "Cheap deal available. Pay SGD {amount} now to secure item. {urgency}.",
        "Limited stock sale. Transfer SGD {amount} immediately. {urgency}.",
        "Order confirmed. Pay SGD {amount} deposit to release goods. {urgency}.",
        "Flash sale ends tonight. Pay SGD {amount} to lock in price. {urgency}.",
        "Seller requires SGD {amount} deposit before shipping. {urgency}.",
    ],
    "bank_impersonation": [
        "Your {bank} account will be frozen unless you verify. {urgency}.",
        "Suspicious {activity} detected in your {bank} account. {urgency}.",
        "{bank} security team detected unauthorised access. {urgency}.",
        "Your {bank} {service} requires re-verification. {urgency}.",
        "{bank} alert: new device login attempt blocked. Confirm identity. {urgency}.",
    ],
    "government_impersonation": [
        "Official government notice: action required immediately. {urgency}.",
        "Verify your identity with the government portal. Avoid {penalty}. {urgency}.",
        "ICA requires your documents for verification. {urgency}.",
        "MOM notice: your work pass status is under review. {urgency}.",
        "IRAS tax discrepancy found in your account. Pay SGD {amount} now. {urgency}.",
    ],
    "fake_friend": [
        "Hi, changed number. This is me. Transfer SGD {amount} urgently. {urgency}.",
        "Lost my phone. Please send SGD {amount} to this new number. {urgency}.",
        "It's me, in trouble. Need SGD {amount} now. Will repay tomorrow. {urgency}.",
        "New number. Family emergency. Transfer SGD {amount} via {platform}. {urgency}.",
        "Hi Mum, it's me. Stuck overseas. Send SGD {amount} please. {urgency}.",
    ],
    "parcel_delivery": [
        "Your parcel cannot be delivered. Pay SGD {amount} fee to reschedule. {urgency}.",
        "Delivery failed. Pay SGD {amount} to release your parcel. {urgency}.",
        "Package held at customs. SGD {amount} clearance fee required. {urgency}.",
        "Your order from overseas is held. Pay SGD {amount} to proceed. {urgency}.",
        "Parcel returned to sender. Update address and pay SGD {amount}. {urgency}.",
    ],
    "rental": [
        "Room available in SG. Pay SGD {amount} deposit immediately. {urgency}.",
        "Cheap rental. Transfer SGD {amount} booking fee now to reserve. {urgency}.",
        "HDB room for rent. Pay SGD {amount} to hold unit. {urgency}.",
        "Landlord requires SGD {amount} deposit via {platform}. {urgency}.",
        "Last unit at this price. Reserve with SGD {amount} now. {urgency}.",
    ],
    "loan": [
        "Fast loan of SGD {amount} approved instantly. {urgency}.",
        "No credit check loan available. Get SGD {amount} today. {urgency}.",
        "Emergency cash SGD {amount} disbursed within 1 hour. {urgency}.",
        "Licensed moneylender. SGD {amount} loan. Low interest. {urgency}.",
        "Approved for SGD {amount} personal loan. Collect now. {urgency}.",
    ],
    "charity": [
        "Donate SGD {amount} to help disaster victims. {urgency}.",
        "Urgent relief fund. SGD {amount} donation needed today. {urgency}.",
        "Children's charity drive. Contribute SGD {amount} via {platform}. {urgency}.",
        "Food bank running low. Donate SGD {amount} to help families. {urgency}.",
        "Cancer patient needs SGD {amount} for treatment. {urgency}.",
    ],
    "prize": [
        "You won SGD {amount} in the lucky draw. Claim now. {urgency}.",
        "Lucky winner selected. Receive SGD {amount} prize. {urgency}.",
        "Congratulations! SGD {amount} reward waiting for you. {urgency}.",
        "You are selected for a SGD {amount} cash prize. {urgency}.",
        "NTUC voucher SGD {amount} unclaimed. Collect before it expires. {urgency}.",
    ],
}

# 150 hand-crafted Singlish templates
SINGLISH_TEMPLATES = [
    # bank / phishing (15)
    "Bro your {bank} account kena {action} already lah.",
    "Eh {bank} got issue leh, must verify now sia.",
    "Wah your {bank} account kena flagged, click link leh.",
    "Siao ah, your {bank} account got problem, faster check.",
    "Your {bank} kena hacked leh, verify now before too late.",
    "Wah {bank} send notice say your {service} blocked already.",
    "Your {bank} kena {action} sia, must verify before midnight.",
    "Eh CPF got issue leh, verify now or account kena locked.",
    "Aiyoh {bank} say got {activity} on your account, faster check.",
    "{bank} alert leh, new device login detected, confirm identity.",
    "Your {bank} {service} kena suspended, respond within 24 hours.",
    "Got {activity} at 3am on your {bank} account sia, faster check.",
    "Siao, your {service} kena flagged leh, click to verify now.",
    "Wah {bank} say got unauthorised {activity}, must confirm now.",
    "Eh your {bank} account will be frozen tomorrow if no verify.",
    # job scam (15)
    "Got easy job one, earn SGD {amount} {frequency} sia.",
    "Part-time job leh, work from home, earn SGD {amount} {frequency}.",
    "No need experience one, just do {role}, earn SGD {amount} daily.",
    "{platform} job sia, do task earn commission, SGD {amount} {frequency}.",
    "Very legit one, earn SGD {amount} per day, no experience needed.",
    "Wah easy money leh, SGD {amount} {frequency}, just do simple task.",
    "Aiya so easy one, {role} job, earn SGD {amount} from home.",
    "Telegram task job, earn SGD {amount} {frequency}, very steady.",
    "Online job leh, SGD {amount} {frequency}, no boss, work anytime.",
    "Oi {platform} has new part-time job, SGD {amount} {frequency}, easy.",
    "Survey job leh, 10 mins only, earn SGD {amount} per survey.",
    "Work from home lah, earn SGD {amount} {frequency}, very easy one.",
    "Got {role} job available, SGD {amount} daily, apply now lah.",
    "Wah this job very good sia, SGD {amount} {frequency}, from home.",
    "Aiya try first lah, {role}, earn SGD {amount}, no commitment.",
    # investment (10)
    "Aiya investment can earn SGD {amount} fast one.",
    "Eh invest SGD {amount} only, can earn back double in one month sia.",
    "Bro trust me, invest SGD {amount} and see the returns.",
    "Wah guaranteed returns leh, put in SGD {amount} only.",
    "Crypto group lah, invest SGD {amount} earn SGD {amount} back.",
    "Very safe investment one, SGD {amount} in, double out.",
    "Uncle already earn SGD {amount} liao, you try also lah.",
    "Limited slots leh, invest SGD {amount} before full.",
    "Passive income sia, SGD {amount} per {frequency}, just invest.",
    "Aiyoh don't miss lah, SGD {amount} investment, confirm earn.",
    # ecommerce (10)
    "Aiya so cheap one, pay SGD {amount} only, item confirm yours.",
    "Flash sale leh, only SGD {amount}, must pay now cannot wait.",
    "Your item seller need SGD {amount} deposit before ship one.",
    "Wah limited stock leh, pay SGD {amount} now to chope.",
    "Last unit sia, SGD {amount} only, transfer now.",
    "Cheap cheap lah, SGD {amount} for this item, grab fast.",
    "Seller say must pay SGD {amount} deposit first, then ship.",
    "Tonight last chance leh, SGD {amount} deal, confirm or not?",
    "Aiya buy now lah, SGD {amount} only, tomorrow price go up.",
    "Very good deal sia, SGD {amount}, pay now to secure.",
    # parcel delivery (10)
    "Parcel cannot deliver lah, pay SGD {amount} first.",
    "Your parcel stuck leh, need SGD {amount} to release.",
    "Eh your parcel reach already but need pay SGD {amount} customs leh.",
    "Package arrived but need SGD {amount} to release from warehouse.",
    "Parcel stuck at customs lah, pay SGD {amount} then can deliver.",
    "Wah your parcel cannot come in leh, SGD {amount} clearance fee.",
    "Delivery failed sia, pay SGD {amount} to reschedule.",
    "Your overseas order stuck lah, SGD {amount} to proceed.",
    "Aiyoh parcel returned leh, update address, pay SGD {amount}.",
    "Package cannot deliver, pay SGD {amount} or they send back.",
    # fake friend (10)
    "Oi transfer SGD {amount} to me first, I pay back tomorrow one.",
    "Eh bro, changed number already lah. Transfer SGD {amount} first can?",
    "Mum ah, new number leh, need SGD {amount} transfer first.",
    "Bro I need SGD {amount} urgently, lost wallet liao, help first.",
    "Aiya stuck lah, can transfer SGD {amount} first? Pay back one.",
    "Eh it's me, new number, family emergency, send SGD {amount}.",
    "Lost phone liao, new number, please send SGD {amount} now.",
    "Wah in trouble leh, need SGD {amount} first, confirm pay back.",
    "Hi, changed number. Send SGD {amount} via {platform} first.",
    "Bro overseas now, need SGD {amount} urgent, no cash here.",
    # rental (10)
    "Wah cheap room available, pay SGD {amount} deposit first lah.",
    "Room very cheap in {area}, pay deposit SGD {amount} first.",
    "HDB room very cheap leh, but must pay SGD {amount} today.",
    "Landlord need SGD {amount} deposit via {platform} first.",
    "Last unit at this price leh, pay SGD {amount} to reserve.",
    "Aiyoh cheap rental sia, SGD {amount} deposit, confirm yours.",
    "Good room in {area}, SGD {amount} only, transfer now.",
    "Room available, very cheap leh, pay SGD {amount} to chope.",
    "Wah nice unit, SGD {amount} deposit, many people want leh.",
    "Landlord say transfer SGD {amount} by tonight, or give to others.",
    # loan (10)
    "Fast loan leh, SGD {amount} approve within 1 hour one.",
    "No credit check one, loan SGD {amount} today collect tomorrow.",
    "Emergency cash lah, SGD {amount} within 1 hour, legit one.",
    "Aiya need money fast? SGD {amount} loan, no check one.",
    "Licensed lender leh, SGD {amount} loan, low interest.",
    "Wah fast approve sia, SGD {amount} loan, collect today.",
    "Personal loan lah, SGD {amount}, apply now respond fast.",
    "Got loan SGD {amount} for you liao, claim before expire.",
    "Quick cash leh, SGD {amount}, no paperwork one.",
    "Aiyoh try lah, SGD {amount} loan, easy to get.",
    # government impersonation (10)
    "Government say must verify your account by today leh.",
    "Government portal leh, verify identity or kena {penalty}.",
    "IRAS say you owe tax leh, pay SGD {amount} or kena {penalty}.",
    "ICA leh, your documents need to verify, urgent.",
    "MOM notice sia, your pass under review, verify now.",
    "Wah government letter leh, must respond by today.",
    "Singpass leh, must re-verify or account kena {action}.",
    "CPF say got issue with your account, verify immediately lah.",
    "Government portal flag your account leh, confirm identity.",
    "Aiyoh official notice lah, action required by today.",
    # prize (10)
    "Eh your prize SGD {amount} still unclaimed leh, quickly take.",
    "Win SGD {amount} voucher leh, just click link and claim.",
    "Lucky draw win SGD {amount} sia, claim now lah.",
    "Wah free cash SGD {amount} from government, go claim now.",
    "Win SGD {amount} in lucky draw lah, last day to claim today.",
    "Congratulations sia, SGD {amount} prize, claim before expire.",
    "Your SGD {amount} reward unclaimed leh, click to receive.",
    "NTUC voucher SGD {amount} for you, claim now lah.",
    "Wah selected as winner sia, SGD {amount}, quick take.",
    "Aiya don't waste lah, SGD {amount} prize, just click.",
    # charity (10)
    "Aiya donate lah, only SGD {amount}, help the poor people.",
    "Donate SGD {amount} lah, help cancer patient, very urgent.",
    "Urgent relief leh, donate SGD {amount} via {platform}.",
    "Food bank need help lah, SGD {amount} donation urgent.",
    "Children charity leh, SGD {amount} only, help them.",
    "Wah very urgent sia, donate SGD {amount} to help families.",
    "Aiyoh help lah, SGD {amount} only, change someone's life.",
    "Disaster victims need help, donate SGD {amount} now lah.",
    "Cancer fund leh, SGD {amount}, help this patient.",
    "Eh donate lah, SGD {amount} via {platform}, very easy.",
]

# NLLB language codes
LANG_MAP = {
    "en": "eng_Latn",
    "ms": "zsm_Latn",
    "ta": "tam_Taml",
    "zh": "zho_Hans",
}

print(f"EN templates:       {sum(len(v) for v in TEMPLATES.values())} across {len(TEMPLATES)} scam types")
print(f"Singlish templates: {len(SINGLISH_TEMPLATES)}")
print(f"Slot dimensions:    {len(SLOTS)}")


EN templates:       60 across 12 scam types
Singlish templates: 120
Slot dimensions:    12


## 5. Generate English and Singlish

In [5]:
def fill_slots(template):
    # two passes to resolve nested slots e.g. {penalty} -> 'SGD {fine} fine'
    text = template
    for _ in range(2):
        for slot, values in SLOTS.items():
            ph = "{" + slot + "}"
            if ph in text:
                text = text.replace(ph, random.choice(values))
    return text


def generate_en(n=1000):
    data = []
    scam_types = list(TEMPLATES.keys())
    for _ in range(n):
        scam = random.choice(scam_types)
        tmpl = random.choice(TEMPLATES[scam])
        data.append({"text": fill_slots(tmpl), "label": 1, "language": "en", "scam_type": scam})
    return pd.DataFrame(data)


def generate_singlish(n=1000):
    data = []
    scam_types = list(TEMPLATES.keys())
    for _ in range(n):
        tmpl = random.choice(SINGLISH_TEMPLATES)
        data.append({
            "text": fill_slots(tmpl),
            "label": 1,
            "language": "singlish",
            "scam_type": random.choice(scam_types),
        })
    return pd.DataFrame(data)


# quick sanity check
test_en = generate_en(200)
test_sg = generate_singlish(200)
print(f"EN  uniqueness (200 samples): {test_en['text'].nunique()} / 200")
print(f"SG  uniqueness (200 samples): {test_sg['text'].nunique()} / 200")
print("\nSample EN:")
print(test_en['text'].head(3).to_string(index=False))
print("\nSample Singlish:")
print(test_sg['text'].head(3).to_string(index=False))


EN  uniqueness (200 samples): 198 / 200
SG  uniqueness (200 samples): 176 / 200

Sample EN:
Donate SGD 50 to help disaster victims. This is...
Suspicious transfer detected in your PayNow acc...
No experience needed. listing assistant earns S...

Sample Singlish:
Wah your parcel cannot come in leh, SGD 150 cle...
Lost phone liao, new number, please send SGD 20...
Wah nice unit, SGD 5000 deposit, many people wa...


## 6. Full English & Singlish generation

In [6]:
N_TARGET = 5000

# english — oversample then dedup
print("Generating English...")
df_en_raw = generate_en(15000)
df_en = df_en_raw.drop_duplicates(subset=["text"]).reset_index(drop=True)
if len(df_en) >= N_TARGET:
    df_en = df_en.head(N_TARGET)
print(f"EN: {len(df_en):,} unique rows")

# singlish — oversample then dedup (ceiling ~3,646 due to finite template pool)
print("Generating Singlish...")
df_sg_raw = generate_singlish(25000)
df_sg = df_sg_raw.drop_duplicates(subset=["text"]).reset_index(drop=True)
if len(df_sg) >= N_TARGET:
    df_sg = df_sg.head(N_TARGET)
print(f"SG: {len(df_sg):,} unique rows (template ceiling expected ~3,646 if < 5000)")


Generating English...
EN: 5,000 unique rows
Generating Singlish...
SG: 3,704 unique rows (template ceiling expected ~3,646 if < 5000)


## 7. Load NLLB-200 and translate to MS / TA / ZH

In [7]:
# load model
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on: {device}")


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded on: cpu


In [8]:
def translate_batch(texts, src, tgt, batch_size=32):
    tokenizer.src_lang = LANG_MAP[src]
    tgt_id = tokenizer.convert_tokens_to_ids(LANG_MAP[tgt])
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f"en -> {tgt}"):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=128).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=tgt_id, max_length=128)
        results.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return results


en_texts = df_en["text"].tolist()
print(f"Translating {len(en_texts)} EN strings to 3 languages...")

# malay
df_ms = df_en.copy()
df_ms["text"] = translate_batch(en_texts, "en", "ms")
df_ms["language"] = "ms"
df_ms = df_ms.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"MS: {len(df_ms):,} unique (collision rate: {(1 - len(df_ms)/len(en_texts))*100:.1f}%)")

# tamil
df_ta = df_en.copy()
df_ta["text"] = translate_batch(en_texts, "en", "ta")
df_ta["language"] = "ta"
df_ta = df_ta.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"TA: {len(df_ta):,} unique (collision rate: {(1 - len(df_ta)/len(en_texts))*100:.1f}%)")

# mandarin — fix currency mistranslation after generation
df_zh = df_en.copy()
df_zh["text"] = translate_batch(en_texts, "en", "zh")
df_zh["language"] = "zh"
df_zh = df_zh.drop_duplicates(subset=["text"]).reset_index(drop=True)
df_zh["text"] = df_zh["text"].str.replace("美元", "SGD", regex=False).str.replace("元", "SGD", regex=False)
print(f"ZH: {len(df_zh):,} unique (collision rate: {(1 - len(df_zh)/len(en_texts))*100:.1f}%)")


Translating 5000 EN strings to 3 languages...


en -> ms:   0%|          | 0/157 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

en -> ms: 100%|██████████| 157/157 [48:49<00:00, 18.66s/it]


MS: 4,831 unique (collision rate: 3.4%)


en -> ta: 100%|██████████| 157/157 [1:04:07<00:00, 24.50s/it]


TA: 4,728 unique (collision rate: 5.4%)


en -> zh: 100%|██████████| 157/157 [53:17<00:00, 20.37s/it] 

ZH: 4,992 unique (collision rate: 0.2%)


## 8. Merge all languages and save

In [9]:
# merge
df_synth = pd.concat([df_en, df_sg, df_ms, df_ta, df_zh], ignore_index=True)
df_synth = df_synth.drop_duplicates(subset=["text", "language"]).reset_index(drop=True)
df_synth = df_synth.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final shape: {df_synth.shape}")
print("\nLanguage distribution:")
print(df_synth["language"].value_counts())
print("\nScam type distribution:")
print(df_synth["scam_type"].value_counts())
print("\nAll label=1 check:", (df_synth["label"] == 1).all())

# save
out_path = os.path.join(DATA_DIR, "synthetic_dataset.csv")
df_synth.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\nSaved: {out_path}")
print(f"Size:  {os.path.getsize(out_path)/1e6:.1f} MB")


Final shape: (23255, 4)

Language distribution:
language
en          5000
zh          4992
ms          4831
ta          4728
singlish    3704
Name: count, dtype: int64

Scam type distribution:
scam_type
job_scam                    2362
phishing                    2114
rental                      2040
investment                  2033
charity                     2015
parcel_delivery             1994
fake_friend                 1981
prize                       1969
bank_impersonation          1925
loan                        1901
ecommerce                   1853
government_impersonation    1068
Name: count, dtype: int64

All label=1 check: True

Saved: /kaggle/working/data/synthetic_dataset.csv
Size:  3.1 MB
